In [ ]:
import xgboost
import sklearn

print("XGBoost:", xgboost.__version__)
print("Scikit-learn:", sklearn.__version__)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('diabetes_prediction_dataset.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns

**EDA PERFORM**

In [ ]:
df['gender'].isnull().sum()

In [ ]:
df['gender'].value_counts()

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(x='gender',data=df,hue='diabetes')

In [ ]:
df['age'].isnull().sum()

In [ ]:
sns.displot(df['age'])

In [ ]:
df['age'].describe()

In [ ]:
sns.boxplot(df['age'])

In [ ]:
df['age'].skew()

In [ ]:
df['age'].min()

In [ ]:
df[df['age']<1]

In [ ]:
from math import e
def age_group(age):
    if age < 18:
      return 'Child'
    elif age < 30:
      return 'Young'
    elif age < 45:
      return 'Adult'
    elif age < 60:
      return 'Middle_age'
    else:
      return 'Senior'

In [ ]:
df['Age_group'] = df['age'].apply(age_group)

In [ ]:
df['hypertension'].unique()

In [ ]:
sns.countplot(x='hypertension',data=df,hue='diabetes')

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df['heart_disease'].unique()

In [ ]:
df['smoking_history'].unique()

In [ ]:
df['smoking_history'].value_counts()

In [ ]:
sns.boxplot(df['bmi'])

In [ ]:
df['bmi'].max()

In [ ]:
df['BMI_category'] = pd.cut(df['bmi'],bins= [0,18.5,24.5,30.0,34.9,40.0,100], labels=['Underweight','Normal weight','Overweight', 'Obesity class 1', 'Obesity class 2','Obesity class 3'])

In [ ]:
df.head()

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(x='BMI_category',data=df,hue='diabetes')

In [ ]:
df.isnull().sum()

**Train test split**

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
cat_cols = ['gender','Age_group', 'smoking_history', 'BMI_category'] ## one hot encoding

numeric_scaler =['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']

other_num=  ['hypertension', 'heart_disease']


feature_cols= cat_cols + numeric_scaler + other_num

X= df[feature_cols]
y= df['diabetes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape)


**Preprocessing using column Transformer**

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#cat_col

cat_ohe_pipeline= Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
#numeric_scaler
numeric_scaler_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())

])

preprocessor = ColumnTransformer([
    ('cat_ohe', cat_ohe_pipeline, cat_cols),
    ('numeric_scaler', numeric_scaler_pipeline, numeric_scaler),
    ('other_num', 'passthrough', other_num)
])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, f1_score, recall_score, classification_report, confusion_matrix


lr_pipline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

lr_pipline.fit(X_train, y_train)
lr_pred = lr_pipline.predict(X_test)
lr_pred_train = lr_pipline.predict(X_train)


lr_train_accuracy =accuracy_score(y_train, lr_pred_train)
lr_test_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_roc_auc = roc_auc_score(y_test, lr_pred)

lr_conf_matrix = confusion_matrix(y_test, lr_pred)
lr_classification_report = classification_report(y_test, lr_pred)


print("Train Accuracy:", lr_train_accuracy)
print("Test Accuracy:", lr_test_accuracy)
print("Precision:", lr_precision)
print("Recall:", lr_recall)
print("F1 Score:", lr_f1)
print("ROC AUC Score:", lr_roc_auc)
print("Confusion Matrix:\n", lr_conf_matrix)
print("Classification Report:\n", lr_classification_report)

In [ ]:
from xgboost import XGBClassifier

xg_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(n_estimators=100,
                            max_depth=4,
                            learning_rate=0.1,
                            objective='binary:logistic',
                            random_state=42))
])

xg_model.fit(X_train, y_train)
xg_pred = xg_model.predict(X_test)
xg_pred_train= xg_model.predict(X_train)




xg_train_accuracy =accuracy_score(y_train, xg_pred_train)
xg_test_accuracy = accuracy_score(y_test, xg_pred)
xg_precision = precision_score(y_test, xg_pred)
xg_model_recall = recall_score(y_test, xg_pred)
xg_f1 = f1_score(y_test, xg_pred)

xg_conf_matrix = confusion_matrix(y_test, xg_pred)
xg_classification_report = classification_report(y_test, xg_pred)


print("Train Accuracy:", xg_train_accuracy)
print("Test Accuracy:", xg_test_accuracy)
print("Precision:", xg_precision)
print("Recall:", xg_model_recall)
print("F1 Score:", xg_f1)
print("Confusion Matrix:\n", xg_conf_matrix)
print("Classification Report:\n", xg_classification_report)



In [ ]:
y_probability= xg_model.predict_proba(X_test)[:, 1]

In [ ]:



threshold = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]
for i in threshold:
    y_pred_proba = (y_probability >= i).astype(int)
    print("Threshold:", i)
    print("Accuracy:", accuracy_score(y_test, y_pred_proba))
    print("Precision:", precision_score(y_test, y_pred_proba))
    print("Recall:", recall_score(y_test, y_pred_proba))
    print("F1 Score:", f1_score(y_test, y_pred_proba))


In [ ]:
from sklearn.metrics import confusion_matrix

threshold = 0.25

y_pred_proba = (y_probability >= threshold).astype(int)

conf_matrix = confusion_matrix(y_test, y_pred_proba)

print(conf_matrix)


In [ ]:
print(classification_report(y_test, y_pred_proba))
print(confusion_matrix(y_test, y_pred_proba))
print(accuracy_score(y_test, y_pred_proba))
print(precision_score(y_test, y_pred_proba))
print(recall_score(y_test, y_pred_proba))
print(f1_score(y_test, y_pred_proba))


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np
import pandas as pd

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), 1
):

    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    fold_model = clone(xg_model)

    fold_model.fit(X_tr, y_tr)

    probability = fold_model.predict_proba(X_val)[:, 1]

    # Threshold = 0.25
    y_pred = (probability >= 0.25).astype(int)

    fold_results.append({
        'Fold': fold,
        'Accuracy': accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred),
        'F1': f1_score(y_val, y_pred)
    })

results_df = pd.DataFrame(fold_results)

print(results_df)

print("\nMean Results:")
print(results_df[['Accuracy', 'Precision', 'Recall', 'F1']].mean())

print("\nStandard Deviation:")
print(results_df[['Accuracy', 'Precision', 'Recall', 'F1']].std())

In [ ]:
param_dist = {
    'model__n_estimators': [200, 300, 400, 500, 600],
    'model__max_depth': [3, 4, 5, 6],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 3, 5, 7],
    'model__gamma': [0, 0.1, 0.2, 0.5],
    'model__reg_alpha': [0, 0.01, 0.1, 1],
    'model__reg_lambda': [1, 2, 5, 10]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
random_search = RandomizedSearchCV(
    estimator=xg_model,
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
print(random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_


In [ ]:
probability = best_model.predict_proba(X_test)[:, 1]

y_pred = (probability >= 0.25).astype(int)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, probability))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), 1
):

    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # Tuned pipeline ki copy
    fold_model = clone(best_model)

    # Fold training
    fold_model.fit(X_tr, y_tr)

    # Positive class probability
    probability = fold_model.predict_proba(X_val)[:, 1]

    # Threshold = 0.25
    y_pred = (probability >= 0.25).astype(int)

    fold_results.append({
        'Fold': fold,
        'Accuracy': accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred),
        'F1': f1_score(y_val, y_pred),
        'ROC_AUC': roc_auc_score(y_val, probability)
    })

results_df = pd.DataFrame(fold_results)

print(results_df)

In [ ]:
print("\nMean Results:")
print(
    results_df[
        ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']
    ].mean()
)

print("\nStandard Deviation:")
print(
    results_df[
        ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']
    ].std()
)

In [ ]:
final_model = clone(best_model)

final_model.fit(
    X_train,
    y_train
)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

test_probability = final_model.predict_proba(X_test)[:, 1]

threshold = 0.25

test_pred = (
    test_probability >= threshold
).astype(int)

print(classification_report(y_test, test_pred))

print("ROC-AUC:",
      roc_auc_score(y_test, test_probability))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

In [ ]:
import joblib
final_package = {
    'model': final_model,
    'threshold': 0.25
}

joblib.dump(
    final_package,
    'diabetes_xgboost_final.pkl'
)

print("Model saved successfully!")

In [ ]:
loaded_package = joblib.load(
    'diabetes_xgboost_final.pkl'
)

print(loaded_package.keys())
print("Threshold:", loaded_package['threshold'])